In [ ]:
import sys, time
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.controllers import controller_a, controller_b, controller_plan
from src.dispatch import annual_bill

fc = pd.read_csv(ROOT / "results" / "forecasts.csv", index_col=0, parse_dates=True)
pv = pd.read_csv(ROOT / "data" / "processed" / "pv_hourly.csv", index_col=0, parse_dates=True)["pv_kwh"]
load = pd.read_csv(ROOT / "data" / "processed" / "household_load_hourly.csv",
                   index_col=0, parse_dates=True)["load_kwh"]
spot = fc["price_actual_mwh"]
assert (fc.index == pv.index).all() and (fc.index == load.index).all()
n_days = len(fc) / 24

In [ ]:
t0 = time.perf_counter()
runs = {
    "A": controller_a(pv, load),                                          # no battery
    "B": controller_b(pv, load),                                          # simple rule
    "C": controller_plan(pv, load, fc["price_forecast_mwh"].values),      # SVR forecast
    "C2": controller_plan(pv, load, fc["price_baseline_mwh"].values),     # last-week forecast
    "D": controller_plan(pv, load, spot.values),                          # perfect foresight
}
print(f"Ran 5 controllers in {time.perf_counter() - t0:.1f} s")

Ran 5 controllers in 6.2 s


In [ ]:
rows = []
for name, r in runs.items():
    charge = r["p_battery_kw"].clip(lower=0)
    discharge = (-r["p_battery_kw"]).clip(lower=0)
    surplus = (r["pv_kwh"] - r["load_kwh"]).clip(lower=0)
    balance_error = (r["pv_kwh"] + discharge + r["import_kwh"]
                     - (r["load_kwh"] + charge + r["export_kwh"])).abs().max()
    bill = annual_bill(r["import_kwh"], r["export_kwh"], spot.values, n_days)
    rows.append({"Controller": name, "Bill ($)": bill["total"],
                 "Import (kWh)": r["import_kwh"].sum(), "Export (kWh)": r["export_kwh"].sum(),
                 "Battery charged (kWh)": charge.sum(),
                 "Charged from grid (kWh)": (charge - surplus).clip(lower=0).sum(),
                 "Min SOC": r["soc"].min(), "Max SOC": r["soc"].max(), "Balance error": balance_error})
summary = pd.DataFrame(rows).set_index("Controller")

assert (summary["Balance error"] < 1e-9).all()
assert summary["Min SOC"].min() >= 0 and summary["Max SOC"].max() <= 1
assert summary.loc["B", "Charged from grid (kWh)"] < 1e-6
summary.round(2)

,Bill ($),Import (kWh),Export (kWh),Battery charged (kWh),Charged from grid (kWh),Min SOC,Max SOC,Balance error
Controller,,,,,,,,
A,1179.84,3403.00,5326.17,0.00,0.00,0.0,0.5,0.0
B,643.46,1018.85,2638.35,2687.83,0.00,0.0,1.0,0.0
C,590.75,1095.38,2611.53,3768.67,635.73,0.0,1.0,0.0
C2,639.02,1141.73,2667.85,3693.06,496.92,0.0,1.0,0.0
D,546.13,1138.88,2670.19,3634.21,490.47,0.0,1.0,0.0


In [ ]:
for name, r in runs.items():
    r.round(6).to_csv(ROOT / "results" / f"run_{name}.csv.gz")
print("saved", list(runs))

saved ['A', 'B', 'C', 'C2', 'D']


In [ ]:
for window in [24, 48, 72]:
    r = controller_plan(pv, load, spot.values, window=window)
    total = annual_bill(r["import_kwh"], r["export_kwh"], spot.values, n_days)["total"]
    print(f"Perfect foresight, {window}-hour planning window: ${total:,.2f}")

Perfect foresight, 24-hour planning window: $667.24
Perfect foresight, 48-hour planning window: $546.13
Perfect foresight, 72-hour planning window: $545.77
